In [ ]:
from prophet import Prophet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. PREPARE DATA

df_prophet = df.reset_index().rename(columns={'Date': 'ds', 'Price': 'y'})

# Ensure correct sorting
df_prophet = df_prophet.sort_values('ds')

# 2. TRAIN / TEST SPLIT (IMPORTANT IMPROVEMENT)
train_size = int(len(df_prophet) * 0.8)
train_df = df_prophet.iloc[:train_size]
test_df = df_prophet.iloc[train_size:]

# 3. DEFINE IMPROVED PROPHET MODEL
model = Prophet(
    changepoint_prior_scale=0.1,   # better trend flexibility
    seasonality_mode='multiplicative',  # better for financial data
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)

# Add custom seasonality
model.add_seasonality(name='monthly', period=30.5, fourier_order=8)

# 4. FIT MODEL

model.fit(train_df)

# 5. MAKE FUTURE PREDICTIONS
future = model.make_future_dataframe(periods=len(test_df), freq='D')
forecast = model.predict(future)

# 6. ALIGN TEST RESULTS

predictions = forecast.iloc[-len(test_df):]['yhat'].values
actuals = test_df['y'].values

# 7. EVALUATION METRICS
mae = mean_absolute_error(actuals, predictions)
mse = mean_squared_error(actuals, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(actuals, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# 8. PLOT RESULTS

plt.figure(figsize=(12,6))
plt.plot(test_df['ds'], actuals, label='Actual')
plt.plot(test_df['ds'], predictions, label='Predicted')
plt.title("Improved Prophet Forecast vs Actual")
plt.legend()
plt.show()

# Optional full forecast plot
model.plot(forecast)
plt.show()